# Experiment: 04 3DEP Pseudolabel Transfer QC

Objective:
- Read the point CSV and summary JSON written by `transfer_3dep_labels_to_casals_refh.py`.
- Check label-transfer distance, match-status balance, pseudo-class balance, and simple signal-separation patterns.
- Keep the notebook lightweight by plotting from saved outputs instead of re-running alignment.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
MAX_SCATTER = 100_000
MAX_BOX_PER_GROUP = 50_000
plt.style.use("seaborn-v0_8-whitegrid")


def latest_point_csv(outputs_root: Path) -> Path:
    candidates = list(outputs_root.glob("**/*_casals_3dep_pseudolabeled_points.csv"))
    if not candidates:
        raise FileNotFoundError(f"No point CSV found under {outputs_root.resolve()}")
    return max(candidates, key=lambda p: p.stat().st_mtime)


def sidecar_path(point_csv: Path, suffix: str) -> Path:
    return point_csv.with_name(point_csv.name.replace("_casals_3dep_pseudolabeled_points.csv", suffix))


def sample_per_group(df: pd.DataFrame, group_col: str, max_n: int) -> pd.DataFrame:
    parts = []
    for _, g in df.groupby(group_col, sort=False, observed=False):
        parts.append(g.sample(n=min(len(g), max_n), random_state=SEED))
    if not parts:
        return df.iloc[0:0].copy()
    return pd.concat(parts, ignore_index=True)


OUTPUTS_ROOT = Path("../outputs")
POINT_CSV = latest_point_csv(OUTPUTS_ROOT)
SUMMARY_CSV = sidecar_path(POINT_CSV, "_label_transfer_summary_by_group.csv")
SUMMARY_JSON = sidecar_path(POINT_CSV, "_alignment_and_transfer_summary.json")

USECOLS = [
    "match_status",
    "pseudo_3dep_class",
    "nearest_3dep_distance_m",
    "refh_snr",
    "refh_amp",
    "bg_mean",
]
df = pd.read_csv(POINT_CSV, usecols=USECOLS)
summary_rows = pd.read_csv(SUMMARY_CSV)
payload = json.loads(SUMMARY_JSON.read_text(encoding="utf-8"))

print("POINT_CSV:", POINT_CSV.resolve())
print("SUMMARY_JSON:", SUMMARY_JSON.resolve())
print("rows:", len(df))
print("alignment dx/dy/dz:", payload["alignment"]["dx_m"], payload["alignment"]["dy_m"], payload["alignment"]["dz_m"])


## Plan

- Hypothesis: strict pseudo-labels should have shorter nearest 3DEP distance than weak or no-match points.
- Hypothesis: no-reference-match points should skew toward lower SNR / lower amplitude than strict pseudo-labels.
- Metrics to record: match-status counts, pseudo-class counts, nearest-distance quantiles, and simple SNR / amp / background contrasts.


In [ ]:
distance = pd.to_numeric(df["nearest_3dep_distance_m"], errors="coerce")
distance = distance.replace([np.inf, -np.inf], np.nan)
status_counts = df["match_status"].value_counts()
pseudo_counts = df["pseudo_3dep_class"].value_counts().sort_values(ascending=False)

quick_metrics = pd.DataFrame(
    {
        "metric": [
            "n_points",
            "alignment_dx_m",
            "alignment_dy_m",
            "alignment_dz_m",
            "nearest_distance_median",
            "nearest_distance_p95",
            "strict_plus_weak_fraction",
            "no_match_fraction",
        ],
        "value": [
            len(df),
            payload["alignment"]["dx_m"],
            payload["alignment"]["dy_m"],
            payload["alignment"]["dz_m"],
            float(distance.median()),
            float(distance.quantile(0.95)),
            float((status_counts.get("strict_pseudolabel", 0) + status_counts.get("weak_pseudolabel", 0)) / max(len(df), 1)),
            float(status_counts.get("no_reference_match", 0) / max(len(df), 1)),
        ],
    }
).set_index("metric")
quick_metrics


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(distance.dropna(), bins=120, color="tab:blue", alpha=0.85)
ax.set_title("nearest_3dep_distance_m histogram")
ax.set_xlabel("meters")
ax.set_ylabel("count")
fig.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
status_counts.sort_values(ascending=False).plot.bar(ax=axes[0], color="tab:green")
axes[0].set_title("match_status counts")
axes[0].set_xlabel("")
axes[0].set_ylabel("count")

pseudo_counts.head(12).plot.bar(ax=axes[1], color="tab:orange")
axes[1].set_title("pseudo_3dep_class counts")
axes[1].set_xlabel("pseudo_3dep_class")
axes[1].set_ylabel("count")
fig.tight_layout()


In [ ]:
top_classes = pseudo_counts.head(6).index.tolist()
box_df = df.loc[df["pseudo_3dep_class"].isin(top_classes), ["pseudo_3dep_class", "refh_snr"]].dropna()
box_df = sample_per_group(box_df, "pseudo_3dep_class", MAX_BOX_PER_GROUP)
box_df["pseudo_3dep_class"] = box_df["pseudo_3dep_class"].astype(str)

fig, ax = plt.subplots(figsize=(9, 4.5))
box_df.boxplot(column="refh_snr", by="pseudo_3dep_class", ax=ax, grid=False)
ax.set_title("refh_snr by pseudo class")
ax.set_xlabel("pseudo_3dep_class")
ax.set_ylabel("refh_snr")
fig.suptitle("")
fig.tight_layout()


In [ ]:
scatter_df = df[["refh_amp", "refh_snr"]].dropna()
if len(scatter_df) > MAX_SCATTER:
    scatter_df = scatter_df.sample(MAX_SCATTER, random_state=SEED)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(scatter_df["refh_amp"], scatter_df["refh_snr"], s=3, alpha=0.15, color="tab:purple", edgecolors="none")
ax.set_title("refh_amp vs refh_snr scatter")
ax.set_xlabel("refh_amp")
ax.set_ylabel("refh_snr")
fig.tight_layout()


In [ ]:
compare = df.loc[df["match_status"].isin(["no_reference_match", "strict_pseudolabel"]), ["match_status", "refh_snr", "refh_amp", "bg_mean"]].copy()
compare = compare.dropna(how="all", subset=["refh_snr", "refh_amp", "bg_mean"])
compare = sample_per_group(compare, "match_status", MAX_BOX_PER_GROUP)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, col in zip(axes, ["refh_snr", "refh_amp", "bg_mean"]):
    compare.boxplot(column=col, by="match_status", ax=ax, grid=False)
    ax.set_title(f"{col} by match status")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=15)
fig.suptitle("")
fig.tight_layout()


## Results

- Use `quick_metrics`, the histogram, and the status / class count plots for a first QC pass.
- Compare the no-match versus strict boxplots to see whether weak-signal points cluster in the no-reference bucket.
- If the latest auto-discovered CSV is not the run you want, set `POINT_CSV` manually in the setup cell and re-run from the top.


## Next steps

- Add per-track or per-sweep small multiples if pseudo-label quality appears spatially heterogeneous.
- Add class-specific distance plots if one transferred class looks unstable.
